# Tuning-curve comparison

Prepare HD-class-3 and 2-D-selected RF curves with `prep_hd_tc` and `prep_rf_tc` using mouse, date, probe, and session IDs. Existing comparison CSVs are reused; missing CSVs are prepared from the saved sources without rerunning RF detection.

Both helpers return normalized curves with all selected units. RF zero-bin selection happens on the native grid before resampling. Each block then selects shared units for the existing paired plots. `rf_only` chooses the RF-only or all-row projection, and `concat` only stacks curves. The pooled comparison resamples m14 to the common grid and normalizes again afterward.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
from pathlib import Path
from textwrap import fill
from IPython.display import display
from matplotlib import pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
from Utils.direction_comparison import (
    resample_profiles, concat, normalize_tc, prepare_comparison,
    plot_comparison_heatmaps, profile_coordinates, tcRange,
)
from Utils.plotting import LIGHT_PLOT_STYLE, plot_direction_comparison
from Utils.statistic_utils import compare_direction_angles
from Utils.tuning_curve_utils import rayleigh_test

root = Path("/mnt/senzailab/Kai/#Recording")
rf_only: bool = True  # Choose the saved RF-only or all-row projection.
rf_response_window_s = (0.0, 0.2)
rf_max_zero_bins = 2  # Skip RF units with > N zero native 1-D bins; None disables.
comparison_bins = 30
plot_options = dict(figsize=(10, 10), vmin=0, vmax=1, colorbar_label="Normalized response")
is_wrap = True


In [ ]:
from Utils.tc_comparison import (
    prep_hd_tc, prep_rf_tc, calculate_tc_statistics, plot_tc_comparison,
)


In [ ]:
mouseid, date, probe = "m14", "260609", "A"
hd_sessionID, rf_sessionID = 1, 3

fm_m14 = prep_hd_tc(mouseid, root, probe, date, hd_sessionID, bins=comparison_bins)
rf_m14 = prep_rf_tc(
    mouseid, root, probe, date, rf_sessionID, bins=comparison_bins,
    rf_range=[-150, 150], time_range_s=rf_response_window_s,
    rf_only=rf_only, max_zero_bins=rf_max_zero_bins,
)

units_m14 = fm_m14.index.intersection(rf_m14.index)
hd_normalized_m14 = fm_m14.loc[units_m14]
rf_normalized_m14 = rf_m14.loc[units_m14]


In [ ]:
mouseid, date = "m15", "260630"
hd_sessionID, rf_sessionID = 1, 3

fm_m15_A = prep_hd_tc(mouseid, root, "A", date, hd_sessionID, bins=comparison_bins)
rf_m15_A = prep_rf_tc(
    mouseid, root, "A", date, rf_sessionID, bins=comparison_bins,
    rf_range=tcRange(True), time_range_s=rf_response_window_s,
    rf_only=rf_only, max_zero_bins=rf_max_zero_bins,
)
fm_m15_B = prep_hd_tc(mouseid, root, "B", date, hd_sessionID, bins=comparison_bins)
rf_m15_B = prep_rf_tc(
    mouseid, root, "B", date, rf_sessionID, bins=comparison_bins,
    rf_range=tcRange(True), time_range_s=rf_response_window_s,
    rf_only=rf_only, max_zero_bins=rf_max_zero_bins,
)

fm_m15 = concat(fm_m15_A, fm_m15_B, label=f"{mouseid} {date} A+B HD")
rf_m15 = concat(rf_m15_A, rf_m15_B, label=f"{mouseid} {date} A+B RF{rf_sessionID}")

units_m15 = fm_m15.index.intersection(rf_m15.index)
hd_normalized_m15 = fm_m15.loc[units_m15]
rf_normalized_m15 = rf_m15.loc[units_m15]


In [ ]:
mouseid, date, probe = "m19", "260827", "A"
hd_sessionID, rf_sessionID = 9, 2

fm_m19 = prep_hd_tc(mouseid, root, probe, date, hd_sessionID, bins=comparison_bins)
rf_m19 = prep_rf_tc(
    mouseid, root, probe, date, rf_sessionID, bins=comparison_bins,
    rf_range=tcRange(True), time_range_s=rf_response_window_s,
    rf_only=rf_only, max_zero_bins=rf_max_zero_bins,
)

units_m19 = fm_m19.index.intersection(rf_m19.index)
hd_normalized_m19 = fm_m19.loc[units_m19]
rf_normalized_m19 = rf_m19.loc[units_m19]


In [ ]:
mouseid, date, probe = "m20", "260921", "A"
hd_sessionID, rf_sessionID = 9, 2

fm_m20 = prep_hd_tc(mouseid, root, probe, date, hd_sessionID, bins=comparison_bins)
rf_m20 = prep_rf_tc(
    mouseid, root, probe, date, rf_sessionID, bins=comparison_bins,
    rf_range=tcRange(True), time_range_s=rf_response_window_s,
    rf_only=rf_only, max_zero_bins=rf_max_zero_bins,
)

units_m20 = fm_m20.index.intersection(rf_m20.index)
hd_normalized_m20 = fm_m20.loc[units_m20]
rf_normalized_m20 = rf_m20.loc[units_m20]


In [ ]:
mouseid, date, probe = "m21", "261006", "A"
hd_sessionID, rf_sessionID = 9, 3

fm_m21 = prep_hd_tc(mouseid, root, probe, date, hd_sessionID, bins=comparison_bins)
rf_m21 = prep_rf_tc(
    mouseid, root, probe, date, rf_sessionID, bins=comparison_bins,
    rf_range=[-180, 180], time_range_s=rf_response_window_s,
    rf_only=rf_only, max_zero_bins=rf_max_zero_bins,
)

units_m21 = fm_m21.index.intersection(rf_m21.index)
hd_normalized_m21 = fm_m21.loc[units_m21]
rf_normalized_m21 = rf_m21.loc[units_m21]


In [ ]:
mouseid, date, probe = "m14", "260609", "A"
hd_sessionID, rf_sessionID = 1, 3

hd = prep_hd_tc(mouseid, root, probe, date, hd_sessionID)
rf = prep_rf_tc(mouseid, root, probe, date, rf_sessionID, rf_range=[-150, 150])

stats = calculate_tc_statistics(hd, rf)
figures = plot_tc_comparison(hd, rf, stats)


In [ ]:
# Native angles, one explicitly selected reference per call.
sorted_fm_m14 = prepare_comparison(hd_normalized_m14, rf_normalized_m14, mode="native")
plot_comparison_heatmaps(sorted_fm_m14, **plot_options)
sorted_rf_m14 = prepare_comparison(rf_normalized_m14, hd_normalized_m14, mode="native")
plot_comparison_heatmaps(sorted_rf_m14, **plot_options)


In [ ]:
# Subtract the reference peak from both curves.
aligned_fm_m14 = prepare_comparison(hd_normalized_m14, rf_normalized_m14, mode="aligned", matched_fill_value=0)
plot_comparison_heatmaps(aligned_fm_m14, **plot_options)
aligned_rf_m14 = prepare_comparison(rf_normalized_m14, hd_normalized_m14, mode="aligned", reference_fill_value=0)
plot_comparison_heatmaps(aligned_rf_m14, **plot_options)


In [ ]:
# Center the reference; add its peak to the matched curve.
summed_fm_m14 = prepare_comparison(hd_normalized_m14, rf_normalized_m14, mode="sum", is_wrap=is_wrap,
                                   matched_fill_value=0)
plot_comparison_heatmaps(summed_fm_m14, **plot_options)
# summed_rf_m14 = prepare_comparison(rf_normalized_m14, hd_normalized_m14, mode="sum", is_wrap=is_wrap, reference_fill_value=0)
# plot_comparison_heatmaps(summed_rf_m14, **plot_options)


In [ ]:
peaks = summed_fm_m14["peaks"]
peak_sum_deg = (peaks["reference_peak_deg"] + peaks["matched_peak_deg"]) % 360
peak_sum_rad = np.deg2rad(peak_sum_deg.dropna().to_numpy())

counts, edges = np.histogram(peak_sum_rad, bins=30, range=(0, 2 * np.pi))
centers_deg = np.rad2deg((edges[:-1] + edges[1:]) / 2)
# Equal occupancy defines a uniform null; retain all bins, including zeros.
rayleigh_score, rayleigh_p = rayleigh_test(counts, np.ones_like(counts), centers_deg)

with plt.rc_context(LIGHT_PLOT_STYLE):
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"}, layout="constrained")
    ax.bar((edges[:-1] + edges[1:]) / 2, counts, width=np.diff(edges), edgecolor="white")
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("Unit count (radial axis)")
    ax.set_title(
        "m14 260609: HD + RF peak angles\n"
        f"n = {int(counts.sum())}\n"
        f"Rayleigh score = {rayleigh_score:.3f}, p = {rayleigh_p:.3g}",
        pad=20,
    )
    plt.show()


In [ ]:
peaks_m14 = sorted_fm_m14["peaks"].dropna(subset=["reference_peak_deg", "matched_peak_deg"])
statistics_m14 = None
if len(peaks_m14) >= 3:
    reference = peaks_m14.reference_peak_deg.to_numpy()
    matched = peaks_m14.matched_peak_deg.to_numpy()
    statistics_m14 = compare_direction_angles(reference, matched, n_permutations=10_000, random_seed=1)
    statistics_figures_m14 = plot_direction_comparison(
        reference, matched, statistics_m14,
        reference_label=fm_m14.attrs["label"], matched_label=rf_m14.attrs["label"],
        reference_ticklabels=profile_coordinates(fm_m14)["ticklabels"],
        matched_ticklabels=profile_coordinates(rf_m14)["ticklabels"],
        show=False,
    )
    for figure, axis in statistics_figures_m14:
        axis.set_title(fill(axis.get_title(), width=60))
        figure.tight_layout()
        display(figure)
        plt.close(figure)


## m15 260630: HD 1 / RF 3

Probes **A + B**, combined with probe identity retained. HD class is selected by `hd_class`.
Load the saved Probe A and Probe B curves, then combine them with `concat`.


In [ ]:
# Native angles, one explicitly selected reference per call.
sorted_fm_m15 = prepare_comparison(hd_normalized_m15, rf_normalized_m15, mode="native")
plot_comparison_heatmaps(sorted_fm_m15, **plot_options)
sorted_rf_m15 = prepare_comparison(rf_normalized_m15, hd_normalized_m15, mode="native")
plot_comparison_heatmaps(sorted_rf_m15, **plot_options)


In [ ]:
# Subtract the reference peak from both curves.
aligned_fm_m15 = prepare_comparison(hd_normalized_m15, rf_normalized_m15, mode="aligned", matched_fill_value=0)
plot_comparison_heatmaps(aligned_fm_m15, **plot_options)
aligned_rf_m15 = prepare_comparison(rf_normalized_m15, hd_normalized_m15, mode="aligned", reference_fill_value=0)
plot_comparison_heatmaps(aligned_rf_m15, **plot_options)


In [ ]:
# Center the reference; add its peak to the matched curve.
summed_fm_m15 = prepare_comparison(hd_normalized_m15, rf_normalized_m15, mode="sum", is_wrap=is_wrap,
                                   matched_fill_value=0)
plot_comparison_heatmaps(summed_fm_m15, **plot_options)
summed_rf_m15 = prepare_comparison(rf_normalized_m15, hd_normalized_m15, mode="sum", is_wrap=is_wrap,
                                   reference_fill_value=0)
plot_comparison_heatmaps(summed_rf_m15, **plot_options)


In [ ]:
peaks = summed_fm_m15["peaks"]
peak_sum_deg = (peaks["reference_peak_deg"] + peaks["matched_peak_deg"]) % 360
peak_sum_rad = np.deg2rad(peak_sum_deg.dropna().to_numpy())

counts, edges = np.histogram(peak_sum_rad, bins=30, range=(0, 2 * np.pi))
centers_deg = np.rad2deg((edges[:-1] + edges[1:]) / 2)
# Equal occupancy defines a uniform null; retain all bins, including zeros.
rayleigh_score, rayleigh_p = rayleigh_test(counts, np.ones_like(counts), centers_deg)

with plt.rc_context(LIGHT_PLOT_STYLE):
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"}, layout="constrained")
    ax.bar((edges[:-1] + edges[1:]) / 2, counts, width=np.diff(edges), edgecolor="white")
    ax.set_theta_zero_location("N")

    ax.set_theta_direction(-1)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("Unit count (radial axis)")
    ax.set_title(
        "m15 260630: HD + RF peak angles\n"
        f"n = {int(counts.sum())}\n"
        f"Rayleigh score = {rayleigh_score:.3f}, p = {rayleigh_p:.3g}",
        pad=20,
    )
    plt.show()


In [ ]:
peaks_m15 = sorted_fm_m15["peaks"].dropna(subset=["reference_peak_deg", "matched_peak_deg"])
statistics_m15 = None
if len(peaks_m15) >= 3:
    reference = peaks_m15.reference_peak_deg.to_numpy()
    matched = peaks_m15.matched_peak_deg.to_numpy()
    statistics_m15 = compare_direction_angles(reference, matched, n_permutations=10_000, random_seed=1)
    statistics_figures_m15 = plot_direction_comparison(
        reference, matched, statistics_m15,
        reference_label=fm_m15.attrs["label"], matched_label=rf_m15.attrs["label"],
        reference_ticklabels=profile_coordinates(fm_m15)["ticklabels"],
        matched_ticklabels=profile_coordinates(rf_m15)["ticklabels"],
        show=False,
    )
    for figure, axis in statistics_figures_m15:
        axis.set_title(fill(axis.get_title(), width=60))
        figure.tight_layout()
        display(figure)
        plt.close(figure)


## m15 260630 Probe A only: HD 1 / RF 3

Use the same sessions and filters as A+B above, selecting Probe **A** only.


In [ ]:
units_m15_A = fm_m15_A.index.intersection(rf_m15_A.index)
hd_normalized_m15_A = fm_m15_A.loc[units_m15_A]
rf_normalized_m15_A = rf_m15_A.loc[units_m15_A]

{"HD": len(fm_m15_A), "RF": len(rf_m15_A), "shared": len(units_m15_A)}


In [ ]:
# Native angles, one explicitly selected reference per call.
sorted_fm_m15_A = prepare_comparison(hd_normalized_m15_A, rf_normalized_m15_A, mode="native")
plot_comparison_heatmaps(sorted_fm_m15_A, **plot_options)
sorted_rf_m15_A = prepare_comparison(rf_normalized_m15_A, hd_normalized_m15_A, mode="native")
plot_comparison_heatmaps(sorted_rf_m15_A, **plot_options)


In [ ]:
# Subtract the reference peak from both curves.
aligned_fm_m15_A = prepare_comparison(hd_normalized_m15_A, rf_normalized_m15_A, mode="aligned", matched_fill_value=0)
plot_comparison_heatmaps(aligned_fm_m15_A, **plot_options)
aligned_rf_m15_A = prepare_comparison(rf_normalized_m15_A, hd_normalized_m15_A, mode="aligned", reference_fill_value=0)
plot_comparison_heatmaps(aligned_rf_m15_A, **plot_options)


In [ ]:
# Center the reference; add its peak to the matched curve.
summed_fm_m15_A = prepare_comparison(hd_normalized_m15_A, rf_normalized_m15_A, mode="sum", is_wrap=is_wrap,
                                     matched_fill_value=0)
plot_comparison_heatmaps(summed_fm_m15_A, **plot_options)
summed_rf_m15_A = prepare_comparison(rf_normalized_m15_A, hd_normalized_m15_A, mode="sum", is_wrap=is_wrap,
                                     reference_fill_value=0)
plot_comparison_heatmaps(summed_rf_m15_A, **plot_options)


In [ ]:
peaks = summed_fm_m15_A["peaks"]
peak_sum_deg = (peaks["reference_peak_deg"] + peaks["matched_peak_deg"]) % 360
peak_sum_rad = np.deg2rad(peak_sum_deg.dropna().to_numpy())

counts, edges = np.histogram(peak_sum_rad, bins=30, range=(0, 2 * np.pi))
centers_deg = np.rad2deg((edges[:-1] + edges[1:]) / 2)
# Equal occupancy defines a uniform null; retain all bins, including zeros.
rayleigh_score, rayleigh_p = rayleigh_test(counts, np.ones_like(counts), centers_deg)

with plt.rc_context(LIGHT_PLOT_STYLE):
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"}, layout="constrained")
    ax.bar((edges[:-1] + edges[1:]) / 2, counts, width=np.diff(edges), edgecolor="white")
    ax.set_theta_zero_location("N")

    ax.set_theta_direction(-1)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("Unit count (radial axis)")
    ax.set_title(
        "m15 260630 Probe A: HD + RF peak angles\n"
        f"n = {int(counts.sum())}\n"
        f"Rayleigh score = {rayleigh_score:.3f}, p = {rayleigh_p:.3g}",
        pad=20,
    )
    plt.show()


In [ ]:
peaks_m15_A = sorted_fm_m15_A["peaks"].dropna(subset=["reference_peak_deg", "matched_peak_deg"])
statistics_m15_A = None
if len(peaks_m15_A) >= 3:
    reference = peaks_m15_A.reference_peak_deg.to_numpy()
    matched = peaks_m15_A.matched_peak_deg.to_numpy()
    statistics_m15_A = compare_direction_angles(reference, matched, n_permutations=10_000, random_seed=1)
    statistics_figures_m15_A = plot_direction_comparison(
        reference, matched, statistics_m15_A,
        reference_label=fm_m15_A.attrs["label"], matched_label=rf_m15_A.attrs["label"],
        reference_ticklabels=profile_coordinates(fm_m15_A)["ticklabels"],
        matched_ticklabels=profile_coordinates(rf_m15_A)["ticklabels"],
        show=False,
    )
    for figure, axis in statistics_figures_m15_A:
        axis.set_title(fill(axis.get_title(), width=60))
        figure.tight_layout()
        display(figure)
        plt.close(figure)


## m15 260630 Probe B only: HD 1 / RF 3

Use the same sessions and filters as A+B above, selecting Probe **B** only.


In [ ]:
units_m15_B = fm_m15_B.index.intersection(rf_m15_B.index)
hd_normalized_m15_B = fm_m15_B.loc[units_m15_B]
rf_normalized_m15_B = rf_m15_B.loc[units_m15_B]

{"HD": len(fm_m15_B), "RF": len(rf_m15_B), "shared": len(units_m15_B)}


In [ ]:
# Native angles, one explicitly selected reference per call.
sorted_fm_m15_B = prepare_comparison(hd_normalized_m15_B, rf_normalized_m15_B, mode="native")
plot_comparison_heatmaps(sorted_fm_m15_B, **plot_options)
sorted_rf_m15_B = prepare_comparison(rf_normalized_m15_B, hd_normalized_m15_B, mode="native")
plot_comparison_heatmaps(sorted_rf_m15_B, **plot_options)


In [ ]:
# Subtract the reference peak from both curves.
aligned_fm_m15_B = prepare_comparison(hd_normalized_m15_B, rf_normalized_m15_B, mode="aligned", matched_fill_value=0)
plot_comparison_heatmaps(aligned_fm_m15_B, **plot_options)
aligned_rf_m15_B = prepare_comparison(rf_normalized_m15_B, hd_normalized_m15_B, mode="aligned", reference_fill_value=0)
plot_comparison_heatmaps(aligned_rf_m15_B, **plot_options)


In [ ]:
# Center the reference; add its peak to the matched curve.
summed_fm_m15_B = prepare_comparison(hd_normalized_m15_B, rf_normalized_m15_B, mode="sum", is_wrap=is_wrap,
                                     matched_fill_value=0)
plot_comparison_heatmaps(summed_fm_m15_B, **plot_options)
summed_rf_m15_B = prepare_comparison(rf_normalized_m15_B, hd_normalized_m15_B, mode="sum", is_wrap=is_wrap,
                                     reference_fill_value=0)
plot_comparison_heatmaps(summed_rf_m15_B, **plot_options)


In [ ]:
peaks = summed_fm_m15_B["peaks"]
peak_sum_deg = (peaks["reference_peak_deg"] + peaks["matched_peak_deg"]) % 360
peak_sum_rad = np.deg2rad(peak_sum_deg.dropna().to_numpy())

counts, edges = np.histogram(peak_sum_rad, bins=30, range=(0, 2 * np.pi))
centers_deg = np.rad2deg((edges[:-1] + edges[1:]) / 2)
# Equal occupancy defines a uniform null; retain all bins, including zeros.
rayleigh_score, rayleigh_p = rayleigh_test(counts, np.ones_like(counts), centers_deg)

with plt.rc_context(LIGHT_PLOT_STYLE):
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"}, layout="constrained")
    ax.bar((edges[:-1] + edges[1:]) / 2, counts, width=np.diff(edges), edgecolor="white")
    ax.set_theta_zero_location("N")

    ax.set_theta_direction(-1)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("Unit count (radial axis)")
    ax.set_title(
        "m15 260630 Probe B: HD + RF peak angles\n"
        f"n = {int(counts.sum())}\n"
        f"Rayleigh score = {rayleigh_score:.3f}, p = {rayleigh_p:.3g}",
        pad=20,
    )
    plt.show()


In [ ]:
peaks_m15_B = sorted_fm_m15_B["peaks"].dropna(subset=["reference_peak_deg", "matched_peak_deg"])
statistics_m15_B = None
if len(peaks_m15_B) >= 3:
    reference = peaks_m15_B.reference_peak_deg.to_numpy()
    matched = peaks_m15_B.matched_peak_deg.to_numpy()
    statistics_m15_B = compare_direction_angles(reference, matched, n_permutations=10_000, random_seed=1)
    statistics_figures_m15_B = plot_direction_comparison(
        reference, matched, statistics_m15_B,
        reference_label=fm_m15_B.attrs["label"], matched_label=rf_m15_B.attrs["label"],
        reference_ticklabels=profile_coordinates(fm_m15_B)["ticklabels"],
        matched_ticklabels=profile_coordinates(rf_m15_B)["ticklabels"],
        show=False,
    )
    for figure, axis in statistics_figures_m15_B:
        axis.set_title(fill(axis.get_title(), width=60))
        figure.tight_layout()
        display(figure)
        plt.close(figure)


## m19 260827: HD 9 / RF 2

Probe **A**. HD class is selected by `hd_class`.


In [ ]:
# Native angles, one explicitly selected reference per call.
sorted_fm_m19 = prepare_comparison(hd_normalized_m19, rf_normalized_m19, mode="native")
plot_comparison_heatmaps(sorted_fm_m19, **plot_options)
sorted_rf_m19 = prepare_comparison(rf_normalized_m19, hd_normalized_m19, mode="native")
plot_comparison_heatmaps(sorted_rf_m19, **plot_options)


In [ ]:
# Subtract the reference peak from both curves.
aligned_fm_m19 = prepare_comparison(hd_normalized_m19, rf_normalized_m19, mode="aligned", matched_fill_value=0)
plot_comparison_heatmaps(aligned_fm_m19, **plot_options)
aligned_rf_m19 = prepare_comparison(rf_normalized_m19, hd_normalized_m19, mode="aligned", reference_fill_value=0)
plot_comparison_heatmaps(aligned_rf_m19, **plot_options)


In [ ]:
# Center the reference; add its peak to the matched curve.
summed_fm_m19 = prepare_comparison(hd_normalized_m19, rf_normalized_m19, mode="sum", is_wrap=is_wrap,
                                   matched_fill_value=0)
plot_comparison_heatmaps(summed_fm_m19, **plot_options)
summed_rf_m19 = prepare_comparison(rf_normalized_m19, hd_normalized_m19, mode="sum", is_wrap=is_wrap,
                                   reference_fill_value=0)
plot_comparison_heatmaps(summed_rf_m19, **plot_options)


In [ ]:
peaks = summed_fm_m19["peaks"]
peak_sum_deg = (peaks["reference_peak_deg"] + peaks["matched_peak_deg"]) % 360
peak_sum_rad = np.deg2rad(peak_sum_deg.dropna().to_numpy())

counts, edges = np.histogram(peak_sum_rad, bins=30, range=(0, 2 * np.pi))
centers_deg = np.rad2deg((edges[:-1] + edges[1:]) / 2)
# Equal occupancy defines a uniform null; retain all bins, including zeros.
rayleigh_score, rayleigh_p = rayleigh_test(counts, np.ones_like(counts), centers_deg)

with plt.rc_context(LIGHT_PLOT_STYLE):
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"}, layout="constrained")
    ax.bar((edges[:-1] + edges[1:]) / 2, counts, width=np.diff(edges), edgecolor="white")
    ax.set_theta_zero_location("N")

    ax.set_theta_direction(-1)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("Unit count (radial axis)")
    ax.set_title(
        "m19 260827: HD + RF peak angles\n"
        f"n = {int(counts.sum())}\n"
        f"Rayleigh score = {rayleigh_score:.3f}, p = {rayleigh_p:.3g}",
        pad=20,
    )
    plt.show()


In [ ]:
peaks_m19 = sorted_fm_m19["peaks"].dropna(subset=["reference_peak_deg", "matched_peak_deg"])
statistics_m19 = None
if len(peaks_m19) >= 3:
    reference = peaks_m19.reference_peak_deg.to_numpy()
    matched = peaks_m19.matched_peak_deg.to_numpy()
    statistics_m19 = compare_direction_angles(reference, matched, n_permutations=10_000, random_seed=1)
    statistics_figures_m19 = plot_direction_comparison(
        reference, matched, statistics_m19,
        reference_label=fm_m19.attrs["label"], matched_label=rf_m19.attrs["label"],
        reference_ticklabels=profile_coordinates(fm_m19)["ticklabels"],
        matched_ticklabels=profile_coordinates(rf_m19)["ticklabels"],
        show=False,
    )
    for figure, axis in statistics_figures_m19:
        axis.set_title(fill(axis.get_title(), width=60))
        figure.tight_layout()
        display(figure)
        plt.close(figure)


## m20 260921: HD 9 / RF 2

Probe **A**. HD class is selected by `hd_class`.


In [ ]:
# Native angles, one explicitly selected reference per call.
sorted_fm_m20 = prepare_comparison(hd_normalized_m20, rf_normalized_m20, mode="native")
plot_comparison_heatmaps(sorted_fm_m20, **plot_options)
sorted_rf_m20 = prepare_comparison(rf_normalized_m20, hd_normalized_m20, mode="native")
plot_comparison_heatmaps(sorted_rf_m20, **plot_options)


In [ ]:
# Subtract the reference peak from both curves.
aligned_fm_m20 = prepare_comparison(hd_normalized_m20, rf_normalized_m20, mode="aligned", matched_fill_value=0)
plot_comparison_heatmaps(aligned_fm_m20, **plot_options)
aligned_rf_m20 = prepare_comparison(rf_normalized_m20, hd_normalized_m20, mode="aligned", reference_fill_value=0)
plot_comparison_heatmaps(aligned_rf_m20, **plot_options)


In [ ]:
# Center the reference; add its peak to the matched curve.
summed_fm_m20 = prepare_comparison(hd_normalized_m20, rf_normalized_m20, mode="sum", is_wrap=is_wrap,
                                   matched_fill_value=0)
plot_comparison_heatmaps(summed_fm_m20, **plot_options)
summed_rf_m20 = prepare_comparison(rf_normalized_m20, hd_normalized_m20, mode="sum", is_wrap=is_wrap,
                                   reference_fill_value=0)
plot_comparison_heatmaps(summed_rf_m20, **plot_options)


In [ ]:
peaks = summed_fm_m20["peaks"]
peak_sum_deg = (peaks["reference_peak_deg"] + peaks["matched_peak_deg"]) % 360
peak_sum_rad = np.deg2rad(peak_sum_deg.dropna().to_numpy())

counts, edges = np.histogram(peak_sum_rad, bins=30, range=(0, 2 * np.pi))
centers_deg = np.rad2deg((edges[:-1] + edges[1:]) / 2)
# Equal occupancy defines a uniform null; retain all bins, including zeros.
rayleigh_score, rayleigh_p = rayleigh_test(counts, np.ones_like(counts), centers_deg)

with plt.rc_context(LIGHT_PLOT_STYLE):
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"}, layout="constrained")
    ax.bar((edges[:-1] + edges[1:]) / 2, counts, width=np.diff(edges), edgecolor="white")
    ax.set_theta_zero_location("N")

    ax.set_theta_direction(-1)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("Unit count (radial axis)")
    ax.set_title(
        "m20 260921: HD + RF peak angles\n"
        f"n = {int(counts.sum())}\n"
        f"Rayleigh score = {rayleigh_score:.3f}, p = {rayleigh_p:.3g}",
        pad=20,
    )
    plt.show()


In [ ]:
peaks_m20 = sorted_fm_m20["peaks"].dropna(subset=["reference_peak_deg", "matched_peak_deg"])
statistics_m20 = None
if len(peaks_m20) >= 3:
    reference = peaks_m20.reference_peak_deg.to_numpy()
    matched = peaks_m20.matched_peak_deg.to_numpy()
    statistics_m20 = compare_direction_angles(reference, matched, n_permutations=10_000, random_seed=1)
    statistics_figures_m20 = plot_direction_comparison(
        reference, matched, statistics_m20,
        reference_label=fm_m20.attrs["label"], matched_label=rf_m20.attrs["label"],
        reference_ticklabels=profile_coordinates(fm_m20)["ticklabels"],
        matched_ticklabels=profile_coordinates(rf_m20)["ticklabels"],
        show=False,
    )
    for figure, axis in statistics_figures_m20:
        axis.set_title(fill(axis.get_title(), width=60))
        figure.tight_layout()
        display(figure)
        plt.close(figure)


In [ ]:
mouseid, date, probe = "m21", "261006", "A"
hd_sessionID, rf_sessionID = 9, 3

fm_m21 = prep_hd_tc(mouseid, root, probe, date, hd_sessionID, bins=comparison_bins)
rf_m21 = prep_rf_tc(
    mouseid, root, probe, date, rf_sessionID, bins=comparison_bins,
    rf_range=[-180, 180], time_range_s=rf_response_window_s,
    rf_only=rf_only, max_zero_bins=rf_max_zero_bins,
)

units_m21 = fm_m21.index.intersection(rf_m21.index)
hd_normalized_m21 = fm_m21.loc[units_m21]
rf_normalized_m21 = rf_m21.loc[units_m21]


In [ ]:

# Native angles, one explicitly selected reference per call.
sorted_fm_m21 = prepare_comparison(hd_normalized_m21, rf_normalized_m21, mode="native")
plot_comparison_heatmaps(sorted_fm_m21, **plot_options)
sorted_rf_m21 = prepare_comparison(rf_normalized_m21, hd_normalized_m21, mode="native")
plot_comparison_heatmaps(sorted_rf_m21, **plot_options)

# Subtract the reference peak from both curves.
aligned_fm_m21 = prepare_comparison(hd_normalized_m21, rf_normalized_m21, mode="aligned", matched_fill_value=0)
plot_comparison_heatmaps(aligned_fm_m21, **plot_options)
aligned_rf_m21 = prepare_comparison(rf_normalized_m21, hd_normalized_m21, mode="aligned", reference_fill_value=0)
plot_comparison_heatmaps(aligned_rf_m21, **plot_options)

# Center the reference; add its peak to the matched curve.
summed_fm_m21 = prepare_comparison(hd_normalized_m21, rf_normalized_m21, mode="sum", is_wrap=is_wrap,
                                   matched_fill_value=0)
plot_comparison_heatmaps(summed_fm_m21, **plot_options)
summed_rf_m21 = prepare_comparison(rf_normalized_m21, hd_normalized_m21, mode="sum", is_wrap=is_wrap,
                                   reference_fill_value=0)
plot_comparison_heatmaps(summed_rf_m21, **plot_options)

peaks_m21 = sorted_fm_m21["peaks"].dropna(subset=["reference_peak_deg", "matched_peak_deg"])
statistics_m21 = None
if len(peaks_m21) >= 3:
    reference = peaks_m21.reference_peak_deg.to_numpy()
    matched = peaks_m21.matched_peak_deg.to_numpy()
    statistics_m21 = compare_direction_angles(reference, matched, n_permutations=10_000, random_seed=1)
    statistics_figures_m21 = plot_direction_comparison(
        reference, matched, statistics_m21,
        reference_label=fm_m21.attrs["label"], matched_label=rf_m21.attrs["label"],
        reference_ticklabels=profile_coordinates(fm_m21)["ticklabels"],
        matched_ticklabels=profile_coordinates(rf_m21)["ticklabels"],
        show=False,
    )
    for figure, axis in statistics_figures_m21:
        axis.set_title(fill(axis.get_title(), width=60))
        figure.tight_layout()
        display(figure)
        plt.close(figure)


In [ ]:
peaks = summed_fm_m21["peaks"]
peak_sum_deg = (peaks["reference_peak_deg"] + peaks["matched_peak_deg"]) % 360
peak_sum_rad = np.deg2rad(peak_sum_deg.dropna().to_numpy())

counts, edges = np.histogram(peak_sum_rad, bins=30, range=(0, 2 * np.pi))
centers_deg = np.rad2deg((edges[:-1] + edges[1:]) / 2)
# Equal occupancy defines a uniform null; retain all bins, including zeros.
rayleigh_score, rayleigh_p = rayleigh_test(counts, np.ones_like(counts), centers_deg)

with plt.rc_context(LIGHT_PLOT_STYLE):
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"}, layout="constrained")
    ax.bar((edges[:-1] + edges[1:]) / 2, counts, width=np.diff(edges), edgecolor="white")
    ax.set_theta_zero_location("N")

    ax.set_theta_direction(-1)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("Unit count (radial axis)")
    ax.set_title(
        "m21 261006: HD + RF peak angles\n"
        f"n = {int(counts.sum())}\n"
        f"Rayleigh score = {rayleigh_score:.3f}, p = {rayleigh_p:.3g}",
        pad=20,
    )
    plt.show()


## All four mice: m14 + m15 + m19 + m20

Pool the mouse-level tables loaded with `rf_type` above, including both m15 probes, then apply the
same HD3 and RF zero-bin limit and select their shared units explicitly.
Rows are stacked without averaging; `(mouse, date, probe, unit_id)` remains unique.


In [ ]:
fm_all = concat(fm_m14, fm_m15, fm_m19, fm_m20, label="All mice HD")
# Use the common grid for pooling; retain the existing zero padding outside m14 coverage.
rf_m14_common = resample_profiles(rf_m14, range=tcRange(True), bins=comparison_bins, fill_value=0)
rf_all = concat(rf_m14_common, rf_m15, rf_m19, rf_m20, label="All mice RF")

units_all = fm_all.index.intersection(rf_all.index)
hd_normalized_all = normalize_tc(fm_all.loc[units_all])
rf_normalized_all = normalize_tc(rf_all.loc[units_all])


In [ ]:
# Native angles, one explicitly selected reference per call.
sorted_fm_all = prepare_comparison(hd_normalized_all, rf_normalized_all, mode="native")
plot_comparison_heatmaps(sorted_fm_all, **plot_options)
sorted_rf_all = prepare_comparison(rf_normalized_all, hd_normalized_all, mode="native")
plot_comparison_heatmaps(sorted_rf_all, **plot_options)


In [ ]:
# Subtract the reference peak from both curves.
aligned_fm_all = prepare_comparison(hd_normalized_all, rf_normalized_all, mode="aligned", matched_fill_value=0)
plot_comparison_heatmaps(aligned_fm_all, **plot_options)
aligned_rf_all = prepare_comparison(rf_normalized_all, hd_normalized_all, mode="aligned", reference_fill_value=0)
plot_comparison_heatmaps(aligned_rf_all, **plot_options)


In [ ]:
# Center the reference; add its peak to the matched curve.
summed_fm_all = prepare_comparison(hd_normalized_all, rf_normalized_all, mode="sum", is_wrap=is_wrap,
                                   matched_fill_value=0)
plot_comparison_heatmaps(summed_fm_all, **plot_options)
summed_rf_all = prepare_comparison(rf_normalized_all, hd_normalized_all, mode="sum", is_wrap=is_wrap,
                                   reference_fill_value=0)
plot_comparison_heatmaps(summed_rf_all, **plot_options)


In [ ]:
peaks = summed_fm_all["peaks"]
peak_sum_deg = (peaks["reference_peak_deg"] + peaks["matched_peak_deg"]) % 360
peak_sum_rad = np.deg2rad(peak_sum_deg.dropna().to_numpy())

counts, edges = np.histogram(peak_sum_rad, bins=30, range=(0, 2 * np.pi))
centers_deg = np.rad2deg((edges[:-1] + edges[1:]) / 2)
# Equal occupancy defines a uniform null; retain all bins, including zeros.
rayleigh_score, rayleigh_p = rayleigh_test(counts, np.ones_like(counts), centers_deg)

with plt.rc_context(LIGHT_PLOT_STYLE):
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"}, layout="constrained")
    ax.bar((edges[:-1] + edges[1:]) / 2, counts, width=np.diff(edges), edgecolor="white")
    ax.set_theta_zero_location("N")

    ax.set_theta_direction(-1)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("Unit count (radial axis)")
    ax.set_title(
        "All mice: HD + RF peak angles\n"
        f"n = {int(counts.sum())}\n"
        f"Rayleigh score = {rayleigh_score:.3f}, p = {rayleigh_p:.3g}",
        pad=20,
    )
    plt.show()


In [ ]:
peaks_all = sorted_fm_all["peaks"].dropna(subset=["reference_peak_deg", "matched_peak_deg"])
statistics_all = None
if len(peaks_all) >= 3:
    reference = peaks_all.reference_peak_deg.to_numpy()
    matched = peaks_all.matched_peak_deg.to_numpy()
    statistics_all = compare_direction_angles(reference, matched, n_permutations=10_000, random_seed=1)
    statistics_figures_all = plot_direction_comparison(
        reference, matched, statistics_all,
        reference_label=fm_all.attrs["label"], matched_label=rf_all.attrs["label"],
        reference_ticklabels=profile_coordinates(fm_all)["ticklabels"],
        matched_ticklabels=profile_coordinates(rf_all)["ticklabels"],
        show=False,
    )
    for figure, axis in statistics_figures_all:
        axis.set_title(fill(axis.get_title(), width=60))
        figure.tight_layout()
        display(figure)
        plt.close(figure)
